In [ ]:
# Forecast of products demand based on M5 dataset

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path
from catboost import CatBoostRegressor


# Folder of data
DATA_DIR = Path("data/m5")

# Main initial dataframes download
sales = pd.read_csv(DATA_DIR / "sales_train_validation.csv")
calendar = pd.read_csv(DATA_DIR / "calendar.csv")
prices = pd.read_csv(DATA_DIR / "sell_prices.csv")


#### Preparing dataframes for further processing
sales = sales.melt(
    id_vars=['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    var_name="day",value_name = 'sales'
)
sales['day'] = sales['day'].str.split('_').str[1].astype('int64')

calendar['date'] = pd.to_datetime(calendar['date'])
calendar['day'] = range(1, len(calendar) + 1)
s = calendar['day'] - calendar['day'].shift(1)
if (s.max(skipna = True)==1) and (s.min(skipna = True)==1):
     print('calendar describes every day')
else:
    print('missing dates in calendar')

# We don't know the real date of product release in a specific store, but we think that it is the beginning of the first week when we see the price

week_start_sale = pd.DataFrame(prices.groupby(['store_id','item_id'])['wm_yr_wk'].min().reset_index().sort_values(by='wm_yr_wk'))
week_first_day = calendar.groupby(by ='wm_yr_wk')['date'].min().reset_index().rename(columns={'date':'start_sale'})
start_sale = week_start_sale.merge(week_first_day, how = 'left', on='wm_yr_wk').drop(columns = ['wm_yr_wk'])

calendar describes every day


In [21]:
## Enrichment of sales data by Calendar, Days after release, Features of lag and rolling
sales1 = sales.merge(calendar, how = 'right', on='day')
sales1['snap'] = np.where(((sales1['state_id']=='CA') & (sales1['snap_CA']==1)) | ((sales1['state_id']=='WI') & (sales1['snap_WI']==1)) | ((sales1['state_id']=='TX') & (sales1['snap_TX']==1)),1,0)
sales1 = sales1.drop(columns=['snap_TX','snap_WI','snap_CA'])
sales2 = sales1.merge(prices, how='left', on=['item_id','store_id','wm_yr_wk'])

# diff2 = (sales2['date'] - sales2.groupby(['item_id','store_id'])['date'].shift(1)).dt.days
sales3 = sales2.merge(start_sale, how='left', on=['store_id','item_id'])

## Preparing with features
sales3['days_after_release'] = (sales3['date'] - sales3['start_sale']).dt.days
sales3 = sales3[sales3['date']>sales3['start_sale']].drop(columns=['start_sale','wm_yr_wk','weekday']) ## filtering all the data before release of the product. just in case
sales3['week_of_year'] = sales3.date.dt.isocalendar().week
sales3['day_of_month'] = sales3['date'].dt.day

g = sales3.groupby(['item_id', 'store_id'])

sales3['lag1']=g['sales'].shift(1)
sales3['lag7']=g['sales'].shift(7)
sales3['lag28']=g['sales'].shift(28)
sales3['rolling_mean7'] = g['sales'].transform(lambda x: x.shift(1).rolling(7).mean())
sales3['rolling_mean28'] = g['sales'].transform(lambda x: x.shift(1).rolling(28).mean())
sales3['rolling_median28'] = g['sales'].transform(lambda x: x.shift(1).rolling(28).median())
sales3['rolling_std7'] = g['sales'].transform(lambda x: x.shift(1).rolling(7).std())
sales3['rolling_std28'] = g['sales'].transform(lambda x: x.shift(1).rolling(28).std())

sales3['price1'] = (g['sell_price'].transform(lambda x: x.shift(1).rolling(7).median()))
sales3['price7'] = (g['sell_price'].transform(lambda x: x.shift(1).rolling(7).median()))
sales3['price28'] = (g['sell_price'].transform(lambda x: x.shift(1).rolling(28).median()))

sales3['trend'] = sales3['rolling_mean7']/sales3['rolling_mean28']
sales3['discount1'] = sales3['sell_price']/sales3['price1']
sales3['discount7'] = sales3['sell_price']/sales3['price7']
sales3['discount28'] = sales3['sell_price']/sales3['price28']
# sales3 = sales3[sales3['sales']>0]

diff3 = (sales3['date'] - sales3.groupby(['item_id','store_id'])['date'].shift(1)).dt.days
if diff3.min(skipna=True)==1 and diff3.max(skipna=True)==1:
    print('We are ready to go further')
else:
    print('Stop and check. Now you have gaps in the product calendar of repeated dates')

Going further


## We finished with features taken into account. Starting to build seasonal naive baseline

In [22]:
## At first we separate train, validation and test
df_regular =sales3.sort_values(by = ['item_id','store_id','date']).copy()

test_start = df_regular['date'].max() - pd.Timedelta(days=27)
test_df = df_regular[df_regular['date']>=test_start].copy() ## later that test_start

train_df = df_regular[df_regular['date']<test_start].copy() ## before test_start

validation_start = test_start - pd.Timedelta(days=27) ## cutoff for validation
validation_df = train_df[train_df['date'] >= validation_start].copy()

train_window = train_df[
    (train_df['date'] >= validation_start - pd.Timedelta(days=179+27)) & (train_df['date']<validation_start)].copy() ## before validation start but half a year period due to limitation of memory

## Here we determine what features are categorical and numerical, clear NaN in categorical features

cat_features = [
    'item_id', 'dept_id','store_id',
    'event_name_1', 'event_type_1',
]
num_features = [
    'wday','month','year','week_of_year','day_of_month','snap',
    'sell_price','days_after_release',
    'lag1', 'lag7', 'lag28',
    'rolling_mean7', 'rolling_mean28','rolling_median28',
    'rolling_std7','rolling_std28',
    'price1','price7','price28',
    'trend',
    'discount1','discount7','discount28',
    'horizon'
]
feature_cols = cat_features + num_features

item_id 3049
dept_id 7
store_id 10
event_name_1 16
event_type_1 4


In [23]:
### Baseline - seasonal forecast
def seasonal_naive_forecast(
    train_df,
    horizon=28,
    season_length=7
):
    """
    Seasonal Naive forecast для одной временной серии.
    train_df: DataFrame с колонками date и sales

    horizon: горизонт прогноза в днях.
    season_length: длина сезонности. Для недельной сезонности = 7.
    """
    if train_df['item_id'].nunique()==1 and train_df['store_id'].nunique()==1:
        item_id = train_df['item_id'].unique()[0]
        store_id = train_df['store_id'].unique()[0]
        train_df = train_df.sort_values(by ='date' ).copy()
        diff = (train_df['date'] - train_df['date'].shift(1)).dt.days
        last_date = train_df['date'].iloc[-1]
        future_dates = pd.date_range(start=last_date+pd.Timedelta(days = 1), end=last_date + pd.Timedelta(days=horizon))

        if diff.max(skipna=True)==1 and diff.min(skipna=True)==1:

            df = pd.DataFrame([])
            s = pd.Series([])
            df['forecast_dates'] = future_dates
            df['item_id'] = item_id
            df['store_id'] = store_id
            if season_length < horizon:
                while len(s)< horizon:
                    s = pd.concat([s, train_df.tail(season_length)['sales']])
                df['forecast_sales'] = s[0:horizon].reset_index(drop=True)

            elif season_length >= horizon:
                df['forecast_sales'] = train_df.tail(season_length )[0:horizon]['sales']


        else:
            raise ValueError("additional preprocessing is required for the calendar")
    else:
        raise ValueError('Input train_df must contain exactly one item_id and one store_id')

    return df

all_forecast = []
for (item_id, store_id), group in train_df.groupby(['item_id','store_id']):
    forecast_df = seasonal_naive_forecast(
        train_df=group, horizon=28,
        season_length=7)
    all_forecast.append(forecast_df)
forecasts_df = pd.concat(all_forecast, ignore_index=True)


baseline_forecast = test_df.merge(forecasts_df, how='left', left_on=['item_id','store_id','date'], right_on = ['item_id','store_id','forecast_dates'])
print(baseline_forecast[['item_id','store_id','date','sales','forecast_dates','forecast_sales']].head(10))
if baseline_forecast['forecast_sales'].isna().sum() == 0:
        print('OK with NA')
else: print( 'Please Check NA values in forecast sales')

mae = (baseline_forecast['sales'] - baseline_forecast['forecast_sales']).abs().mean()
print('MAE of baseline = ', round(mae,2))
rmse = (((baseline_forecast['sales'] - baseline_forecast['forecast_sales'])**2).mean())**0.5
print('RMSE of baseline = ', round(rmse,2))

       item_id store_id       date  sales forecast_dates forecast_sales
0  FOODS_1_001     CA_1 2016-03-28    2.0     2016-03-28            0.0
1  FOODS_1_001     CA_1 2016-03-29    1.0     2016-03-29            2.0
2  FOODS_1_001     CA_1 2016-03-30    1.0     2016-03-30            2.0
3  FOODS_1_001     CA_1 2016-03-31    0.0     2016-03-31            0.0
4  FOODS_1_001     CA_1 2016-04-01    4.0     2016-04-01            1.0
5  FOODS_1_001     CA_1 2016-04-02    0.0     2016-04-02            1.0
6  FOODS_1_001     CA_1 2016-04-03    0.0     2016-04-03            0.0
7  FOODS_1_001     CA_1 2016-04-04    4.0     2016-04-04            0.0
8  FOODS_1_001     CA_1 2016-04-05    1.0     2016-04-05            2.0
9  FOODS_1_001     CA_1 2016-04-06    3.0     2016-04-06            2.0
OK with NA
MAE of baseline =  1.23
RMSE of baseline =  2.76


In [24]:
## Now we prepare for the modeling. I'm gonna use single model of CatBoost to predict all horizons. So at first we need to
def make_direct_training_data(df, horizon = 28):
    training_parts = []
    if df['item_id'].nunique()==1 and df['store_id'].nunique()==1:
        for hor in range(1, horizon+1):
            tmp = df.copy()
            tmp['horizon'] = hor
            tmp['target'] = tmp['sales'].shift(-hor)
            tmp['target_date'] = tmp['date'].shift(-hor)
            tmp = tmp.dropna(subset=['target'])
            training_parts.append(tmp)
        training_df = pd.concat(training_parts, ignore_index = True)
    else:
        raise ValueError('Input df must contain exactly one item_id and one store_id')
    return training_df


train_help = []
for (item_id, store_id), group in train_window.groupby(['item_id','store_id']):
    tmp = make_direct_training_data(group, horizon = 28)
    train_help.append(tmp)
training_df = pd.concat(train_help, ignore_index = True)
del train_help

print(training_df.head())
print(training_df['horizon'].value_counts().sort_index())
print(training_df['target'].isna().sum())


       item_id  dept_id cat_id store_id state_id   day  sales       date  \
0  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  1679    0.0 2015-09-03   
1  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  1680    0.0 2015-09-04   
2  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  1681    0.0 2015-09-05   
3  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  1682    0.0 2015-09-06   
4  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  1683    0.0 2015-09-07   

   wday  month  ...  price1 price7 price28     trend discount1  discount7  \
0     6      9  ...    2.24   2.24    2.24  0.666667       1.0        1.0   
1     7      9  ...    2.24   2.24    2.24  0.666667       1.0        1.0   
2     1      9  ...    2.24   2.24    2.24  0.235294       1.0        1.0   
3     2      9  ...    2.24   2.24    2.24  0.235294       1.0        1.0   
4     3      9  ...    2.24   2.24    2.24  0.285714       1.0        1.0   

   discount28  horizon  target  target_date  
0         1.0        1     0.0   2

In [16]:
#### We take only nessesary columns from the training dataset. and only 5M of rows due to limitations of memory

X_train = training_df[feature_cols].reset_index(drop = True)
y_train = training_df['target'].reset_index(drop = True)

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_train),size=5_000_000,replace=False)

X_train_sample = X_train.iloc[sample_idx].copy()
y_train_sample = y_train.iloc[sample_idx].copy()

# NaN в категориальных признаках
X_train_sample['event_name_1'] = (X_train_sample['event_name_1'].fillna('None').astype(str))

print(X_train_sample[cat_features].isna().sum())


item_id 3049
dept_id 7
store_id 10
event_name_1 16
event_type_1 4
item_id         0
dept_id         0
store_id        0
event_name_1    0
event_type_1    0
dtype: int64


In [17]:
### Here we prepare features for validation

# Origin_date is the last date when we assume that we know sales. It is the last day of train_window
origin_date = training_df['date'].max()
origins = training_df[training_df['date'] == origin_date].copy()


## Features known for the whole validation period which describe state in the future (prices, holidays...)
future_features = validation_df[
    [   'item_id','dept_id','store_id',
        'date','wday','month','year','week_of_year', 'day_of_month',
        'snap',
        'sell_price',
        'event_name_1','event_type_1',
        'days_after_release']].rename(columns={'date': 'target_date'}).copy()
## Features known for every item*store at the moment of origins (1 raw per each item*store )
history_features = origins[
    [   'item_id','store_id',
        'lag1','lag7','lag28',
        'rolling_mean7','rolling_mean28','rolling_median28',
        'rolling_std7','rolling_std28',
        'price1','price7','price28',
        'trend',
        'discount1','discount7','discount28'].copy()

###### Now we combine the same structure as the dataframe with ll features, but some of them we know at the origin_date, and others are spread in future with known future characteristics
validation_features = future_features.merge(history_features,how='left',on=['item_id', 'store_id'])
## We determine horizon which is needed for the single CatBoost model
validation_features['horizon'] = (validation_features['target_date'] - origin_date).dt.days
X_validation = validation_features[feature_cols].copy()


for col in cat_features:
    X_validation[col] = X_validation[col].fillna('None')
    X_train_sample[col] = X_train_sample[col].fillna('None')

print('X_train_sample:', X_train_sample.shape)
print('X_validation:', X_validation.shape)

In [26]:
### We prepared everything for building a model, so let's go

model = CatBoostRegressor(
    loss_function='RMSE',
    eval_metric='MAE',
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=100)


### The model is learning on train_sample dataset
model.fit(
    X_train_sample,
    y_train_sample,
    cat_features=cat_features)


#####______________________
## We wand to evaluate how does if wot=rk on validation dataframe. We put validations features into the model and add the column to validations_features
validation_features['forecast'] = model.predict(X_validation)


### Now we combine the prediction and the actuals
validation_result = validation_features[['item_id','store_id', 'target_date','horizon','forecast']].merge(
    validation_df[['item_id','store_id','date','sales']],
    how='left', left_on=['item_id','store_id','target_date'], right_on=['item_id','store_id','date'])
validation_result = validation_result.rename(columns={'sales': 'actual'})


### are there any missing actuals in the validation dataframe?
print('Missing actual:',validation_result['actual'].isna().sum())

print(validation_result[[   'item_id', 'store_id','target_date','horizon','actual','forecast']].head(10))

### Metrics of the model
mae = (validation_result['actual']- validation_result['forecast']).abs().mean()
rmse = np.sqrt((validation_result['actual']- validation_result['forecast']) ** 2).mean()

print('Validation MAE =', round(mae, 2))
print('Validation RMSE =', round(rmse, 2))

# 0:	learn: 1.5898424	total: 523ms	remaining: 8m 42s
# 100:	learn: 1.0192004	total: 41.1s	remaining: 6m 5s
# 200:	learn: 1.0047970	total: 1m 24s	remaining: 5m 34s
# 300:	learn: 0.9973280	total: 2m 7s	remaining: 4m 55s
# 400:	learn: 0.9926963	total: 2m 50s	remaining: 4m 15s
# 500:	learn: 0.9878096	total: 3m 32s	remaining: 3m 32s
# 600:	learn: 0.9827985	total: 4m 16s	remaining: 2m 50s
# 700:	learn: 0.9800193	total: 4m 58s	remaining: 2m 7s
# 800:	learn: 0.9778451	total: 5m 41s	remaining: 1m 24s
# 900:	learn: 0.9761370	total: 6m 25s	remaining: 42.4s
# 999:	learn: 0.9742413	total: 7m 8s	remaining: 0us
# Missing actual: 0
#        item_id store_id target_date  horizon  actual  forecast
# 0  FOODS_1_001     CA_1  2016-03-01        1     0.0  0.509819
# 1  FOODS_1_001     CA_1  2016-03-02        2     1.0  0.591379
# 2  FOODS_1_001     CA_1  2016-03-03        3     0.0  0.619176
# 3  FOODS_1_001     CA_1  2016-03-04        4     0.0  0.557436
# 4  FOODS_1_001     CA_1  2016-03-05        5     2.0  0.560594
# 5  FOODS_1_001     CA_1  2016-03-06        6     0.0  0.578043
# 6  FOODS_1_001     CA_1  2016-03-07        7     0.0  0.555704
# 7  FOODS_1_001     CA_1  2016-03-08        8     0.0  0.513186
# 8  FOODS_1_001     CA_1  2016-03-09        9     0.0  0.538213
# 9  FOODS_1_001     CA_1  2016-03-10       10     1.0  0.558088
# Validation MAE = 1.08
# Validation RMSE = 1.08

0:	learn: 1.5898424	total: 523ms	remaining: 8m 42s
100:	learn: 1.0192004	total: 41.1s	remaining: 6m 5s
200:	learn: 1.0047970	total: 1m 24s	remaining: 5m 34s
300:	learn: 0.9973280	total: 2m 7s	remaining: 4m 55s
400:	learn: 0.9926963	total: 2m 50s	remaining: 4m 15s
500:	learn: 0.9878096	total: 3m 32s	remaining: 3m 32s
600:	learn: 0.9827985	total: 4m 16s	remaining: 2m 50s
700:	learn: 0.9800193	total: 4m 58s	remaining: 2m 7s
800:	learn: 0.9778451	total: 5m 41s	remaining: 1m 24s
900:	learn: 0.9761370	total: 6m 25s	remaining: 42.4s
999:	learn: 0.9742413	total: 7m 8s	remaining: 0us
Missing actual: 0
       item_id store_id target_date  horizon  actual  forecast
0  FOODS_1_001     CA_1  2016-03-01        1     0.0  0.509819
1  FOODS_1_001     CA_1  2016-03-02        2     1.0  0.591379
2  FOODS_1_001     CA_1  2016-03-03        3     0.0  0.619176
3  FOODS_1_001     CA_1  2016-03-04        4     0.0  0.557436
4  FOODS_1_001     CA_1  2016-03-05        5     2.0  0.560594
5  FOODS_1_001     CA_

In [27]:
### Now we are looking for the best parameter using grid iterations*depth

iterations_list = [500, 1000, 1500]
depth_list = [6, 8, 10]

results = []

for iterations in iterations_list:
    for depth in depth_list:

        print(f'\nTraining: iterations={iterations}, depth={depth}')

        model = CatBoostRegressor(
            loss_function='RMSE',
            eval_metric='MAE',
            iterations=iterations,
            learning_rate=0.05,
            depth=depth,
            random_seed=42,
            verbose=False
        )

        model.fit(
            X_train_sample,
            y_train_sample,
            cat_features=cat_features
        )

        forecast = model.predict(X_validation)

        mae = (
            validation_result['actual'] - forecast
        ).abs().mean()

        rmse = np.sqrt(
            ((validation_result['actual'] - forecast) ** 2).mean()
        )

        results.append({
            'iterations': iterations,
            'depth': depth,
            'MAE': mae,
            'RMSE': rmse
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values('MAE')

print(results_df)
# Training: iterations=1500, depth=10
#    iterations  depth       MAE      RMSE
# 3        1000      6  1.063566  2.282704
# 6        1500      6  1.065099  2.288695
# 0         500      6  1.065220  2.282283
# 2         500     10  1.066018  2.292545
# 1         500      8  1.068164  2.284326
# 5        1000     10  1.070801  2.304752
# 7        1500      8  1.073315  2.296634
# 8        1500     10  1.074646  2.314555
# 4        1000      8  1.075338  2.289589


Training: iterations=500, depth=6

Training: iterations=500, depth=8

Training: iterations=500, depth=10

Training: iterations=1000, depth=6

Training: iterations=1000, depth=8

Training: iterations=1000, depth=10

Training: iterations=1500, depth=6

Training: iterations=1500, depth=8

Training: iterations=1500, depth=10
   iterations  depth       MAE      RMSE
3        1000      6  1.063566  2.282704
6        1500      6  1.065099  2.288695
0         500      6  1.065220  2.282283
2         500     10  1.066018  2.292545
1         500      8  1.068164  2.284326
5        1000     10  1.070801  2.304752
7        1500      8  1.073315  2.296634
8        1500     10  1.074646  2.314555
4        1000      8  1.075338  2.289589
